# BMC Civic Complaints — Data Cleaning
### Capstone: "Potholes & Priorities"

This notebook takes the raw Kaggle CSV (`bmc_train.csv`) and produces a clean CSV .


In [1]:
import pandas as pd

## Step 1: Loading the raw data



In [2]:
df = pd.read_csv(r"C:\Users\91882\Downloads\bmc_train.csv")
df

,complaint_id,complaint_date,year,month,is_monsoon_season,complaint_time_of_day,ward_code,ward_area,zone,ward_type,...,num_reassignments,complaint_status,contractor_category,work_quality_rating,site_inspected,defect_liability_claim,estimated_cost_inr,infrastructure_age_years,months_since_last_maintained,citizen_satisfied
0,BMC20240942049,2024-09-20,2024,9,1,Evening (5-9pm),A,Colaba-Fort,City,South,...,2,Resolved,Class C (Small),Good,1,0,1500,2,8,1
1,BMC20240088068,2024-01-02,2024,1,0,Daytime (10am-5pm),P/N,Malad,Western,Suburban,...,1,Resolved,Class B (Medium),Fair,0,0,38600,0,3,0
2,BMC20220421176,2022-04-12,2022,4,0,Daytime (10am-5pm),B,Mandvi-Masjid,City,South,...,1,Resolved,Class A (Large),Fair,1,1,13000,10,21,1
3,BMC20240073658,2024-02-28,2024,2,0,Evening (5-9pm),M/E,Govandi-Mankhurd,Eastern,Suburban,...,0,Closed Without Resolution,Class C (Small),Fair,0,0,2400,20,15,0
4,BMC20190483987,2019-03-15,2019,3,0,Daytime (10am-5pm),L,Kurla-Vidyavihar,Eastern,Suburban,...,3,Resolved,Class C (Small),Good,0,0,11400,40,59,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
959995,BMC20180663117,2018-11-25,2018,11,0,Evening (5-9pm),K/W,Andheri West-Juhu,Western,Suburban,...,0,Resolved,Municipal Staff,Poor,0,0,22300,25,11,1
959996,BMC20180996878,2018-01-19,2018,1,0,Night (9pm-6am),A,Colaba-Fort,City,South,...,5,Resolved,Class C (Small),Fair,0,0,24100,30,48,0
959997,BMC20230653237,2023-06-16,2023,6,1,Night (9pm-6am),N,Ghatkopar,Eastern,Suburban,...,2,Closed Without Resolution,Class B (Medium),Good,1,0,2100,15,21,0
959998,BMC20230882954,2023-06-12,2023,6,1,Daytime (10am-5pm),G/N,Dharavi-Matunga,City,Central,...,2,Resolved,Municipal Staff,Fair,1,0,1200,5,21,1


## Step 2: Checking the data types



In [3]:
df.dtypes

complaint_id                    object
complaint_date                  object
year                             int64
month                            int64
is_monsoon_season                int64
complaint_time_of_day           object
ward_code                       object
ward_area                       object
zone                            object
ward_type                       object
population_density              object
ward_slum_percentage             int64
complaint_category              object
department_assigned             object
complaint_channel               object
severity                        object
has_photo_evidence               int64
has_gps_location                 int64
media_attention                  int64
politically_sensitive            int64
complainant_type                object
property_type                   object
repeat_complainant               int64
prior_complaints_count           int64
resolution_days                  int64
num_reassignments        

Noticed that `complaint_date` is read as plain text (`object`), not an actual date.

## Step 3: Fixing data types
1. Converting `complaint_date` from text to a real datetime using `pd.to_datetime()`.
2. Converting repeated text columns (ward, category, severity, etc.) to pandas' `category` dtype 

In [4]:
df["complaint_date"] = pd.to_datetime(df["complaint_date"])

category_cols = [
    "ward_code", "ward_area", "zone", "ward_type", "population_density",
    "complaint_category", "department_assigned", "complaint_channel",
    "severity", "complainant_type", "property_type", "complaint_status",
    "contractor_category", "work_quality_rating", "complaint_time_of_day",
]
for col in category_cols:
    df[col] = df[col].astype("category")

df.dtypes

complaint_id                            object
complaint_date                  datetime64[ns]
year                                     int64
month                                    int64
is_monsoon_season                        int64
complaint_time_of_day                 category
ward_code                             category
ward_area                             category
zone                                  category
ward_type                             category
population_density                    category
ward_slum_percentage                     int64
complaint_category                    category
department_assigned                   category
complaint_channel                     category
severity                              category
has_photo_evidence                       int64
has_gps_location                         int64
media_attention                          int64
politically_sensitive                    int64
complainant_type                      category
property_type

## Step 4: Building columns for KPIs need

The raw data doesn't have every field  needed for the dashboard. We build five new ones:

| New column | What it does |
|---|---|
| `month_name` | Turns month number (1–12) into a readable label like "September" |
| `monsoon_label` | Turns the 0/1 monsoon flag into "Monsoon"/"Non-Monsoon" |
| `backlog_aging_bucket` | Buckets `resolution_days` into 0–7 / 8–30 / 30+ day groups |
| `is_unresolved` | Flags complaints still open: In Progress, Escalated, or Reopened |
| `is_system_failure` | Stricter flag: only Escalated/Reopened (system tried and failed to close it) |

In [5]:
# Readable month name
df["month_name"] = df["complaint_date"].dt.month_name()

# Readable monsoon label
df["monsoon_label"] = df["is_monsoon_season"].replace({1: "Monsoon", 0: "Non-Monsoon"})

# Backlog aging bucket 
df["backlog_aging_bucket"] = pd.cut(
    df["resolution_days"],
    bins=[-1, 7, 30, 10000],
    labels=["0-7 days", "8-30 days", "30+ days"]
)

# Unresolved flag 
df["is_unresolved"] = (
    (df["complaint_status"] == "In Progress") |
    (df["complaint_status"] == "Escalated") |
    (df["complaint_status"] == "Reopened")
).astype(int)

# System failure flag
df["is_system_failure"] = (
    (df["complaint_status"] == "Escalated") |
    (df["complaint_status"] == "Reopened")
).astype(int)

# Checking the result
df[["complaint_date", "month_name", "monsoon_label", "resolution_days",
    "backlog_aging_bucket", "is_unresolved","is_system_failure"]].head(10)

,complaint_date,month_name,monsoon_label,resolution_days,backlog_aging_bucket,is_unresolved,is_system_failure
0,2024-09-20,September,Monsoon,37,30+ days,0,0
1,2024-01-02,January,Non-Monsoon,12,8-30 days,0,0
2,2022-04-12,April,Non-Monsoon,6,0-7 days,0,0
3,2024-02-28,February,Non-Monsoon,1,0-7 days,0,0
4,2019-03-15,March,Non-Monsoon,25,8-30 days,0,0
5,2024-07-28,July,Monsoon,23,8-30 days,0,0
6,2019-03-11,March,Non-Monsoon,57,30+ days,0,0
7,2021-02-27,February,Non-Monsoon,11,8-30 days,1,0
8,2024-09-25,September,Monsoon,1,0-7 days,1,1
9,2019-09-14,September,Monsoon,28,8-30 days,1,1


## Step 5: checking the new columns

In [6]:
print(df["backlog_aging_bucket"].value_counts())
print()
print(df["monsoon_label"].value_counts())

backlog_aging_bucket
8-30 days    510699
0-7 days     364396
30+ days      84905
Name: count, dtype: int64

monsoon_label
Non-Monsoon    639362
Monsoon        320638
Name: count, dtype: int64


## Step 6: Exporting the clean file



In [7]:
OUTPUT_PATH = "bmc_complaints_clean.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(f"Final shape: {df.shape}")

Saved to bmc_complaints_clean.csv
Final shape: (960000, 40)
